# Hyperparameter optimization for ultraheavy diquark ML-based signal detection

In [117]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [118]:
df = pd.read_parquet("/home/gmajeri/notebooks/data_new_features_6_jets.parquet")

df

,jet_multiplicity,combined_invariant_mass,sphericity,aplanarity,centrality,total_energy,total_transverse_momentum,p_T_min,p_T_mean,p_T_stddev,...,chi2_3_mean,chi2_3_max,2jet_invariant_mass_mean,2jet_invariant_mass_stddev,3jet_invariant_mass_mean,3jet_invariant_mass_stddev,6jet_invariant_mass_mean,6jet_invariant_mass_stddev,n_jet_pairs_near_w_mass,label
index,,,,,,,,,,,,,,,,,,,,,
0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0,BKG:gg_bbbar
0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0,BKG:gg_bbbar
4656569659233790776,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0,BKG:gg_bbbar
0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0,BKG:gg_bbbar
4653989647435531168,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0,BKG:gg_bbbar
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,5,7459.883197,0.029608,0.000286,0.560534,7462.769043,4183.134277,60.601192,836.626855,783.006900,...,0.000000,0.000000,1470.769043,1844.404083,3236.501758,2493.996065,0.00000,0.0,1,SIG:Suu
0,6,7475.343691,0.031678,0.002972,0.622338,7484.976562,4658.182129,45.025780,776.363688,836.250963,...,27.526419,27.526419,1150.550130,1549.715758,2503.402734,2215.655467,7475.34375,0.0,0,SIG:Suu
0,5,7067.953118,0.173947,0.059047,0.442356,7367.555176,3259.083008,50.120354,651.816602,375.098078,...,0.000000,0.000000,1808.013477,1314.033207,3596.017188,1433.684340,0.00000,0.0,0,SIG:Suu


In [119]:
X = df.drop(columns="label")
y: pd.Series = df["label"] == "SIG:Suu"

In [120]:
num_signal = (y == 1).sum()
bkg_mask = y == 0
X_bkg = X.iloc[bkg_mask].sample(n=num_signal, replace=False, random_state=7)
y_bkg = y.iloc[bkg_mask].sample(n=num_signal, replace=False, random_state=7)

X_sgn = X.iloc[~bkg_mask]
y_sgn = y.iloc[~bkg_mask]

X_balanced = pd.concat((X_bkg, X_sgn), ignore_index=True).reindex()
y_balanced = pd.concat((y_bkg, y_sgn), ignore_index=True).reindex()

In [121]:
from sklearn.model_selection import StratifiedKFold

# from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

In [122]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

In [123]:
average_precision_scores = np.empty(skf.get_n_splits(), dtype=np.float64)

for i, (train_index, test_index) in enumerate(skf.split(X_balanced, y_balanced)):
    X_train = X_balanced.iloc[train_index]
    y_train = y_balanced.iloc[train_index]
    X_test = X_balanced.iloc[test_index]
    y_test = y_balanced.iloc[test_index]

    print(f"Fold #{i + 1}")
    print(
        f"  We have a total of {(y_train == 0).sum()} backgrounds for training and {(y_test == 0).sum()} for testing"
    )

    train_signals_count = (y_train == 1).sum()
    test_signals_count = (y_test == 1).sum()
    print(
        f"  We're training on {train_signals_count} signals and testing on {test_signals_count} signals"
    )

    # print("  Subsampling backgrounds")

    # bkg_train_mask = y_train == 0

    # X_train_bkg = X_train.iloc[bkg_train_mask].iloc[:train_signals_count]
    # y_train_bkg = y_train.iloc[bkg_train_mask].iloc[:train_signals_count]

    # X_train_sig = X_train.iloc[~bkg_train_mask]
    # y_train_sig = y_train.iloc[~bkg_train_mask]

    # X_train = pd.concat((X_train_bkg, X_train_sig), ignore_index=True)
    # y_train = pd.concat((y_train_bkg, y_train_sig), ignore_index=True)

    # print(f"  Now we're training on {(y_train == 1).sum()} signals and {(y_train == 0).sum()} backgrounds")

    # print("  Fitting logistic regression model...")
    # clf = LogisticRegression(max_iter=1000, random_state=42, l1_ratio=0)
    # clf.fit(X_train, y_train)

    print("  Fitting random forest classifier...")
    clf = RandomForestClassifier(n_jobs=120, random_state=42, verbose=1)
    clf.fit(X_train, y_train)

    y_test_predicted = clf.predict_proba(X_test)
    assert isinstance(y_test_predicted, np.ndarray)

    # First column is probability of being background, second column is probability of being signal
    y_test_predicted = y_test_predicted[:, 1]

    aps = average_precision_score(y_test, y_test_predicted)
    print("Average precision score:", aps)

    average_precision_scores[i] = aps

Fold #1
  We have a total of 74111 backgrounds for training and 18528 for testing
  We're training on 74111 signals and testing on 18528 signals
  Fitting random forest classifier...


[Parallel(n_jobs=120)]: Using backend ThreadingBackend with 120 concurrent workers.
[Parallel(n_jobs=120)]: Done  63 out of 100 | elapsed:    1.1s remaining:    0.7s
[Parallel(n_jobs=120)]: Done 100 out of 100 | elapsed:    1.3s finished
[Parallel(n_jobs=100)]: Using backend ThreadingBackend with 100 concurrent workers.
[Parallel(n_jobs=100)]: Done   2 out of 100 | elapsed:    0.0s remaining:    1.2s
[Parallel(n_jobs=100)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=120)]: Using backend ThreadingBackend with 120 concurrent workers.


Average precision score: 0.9953570358077828
Fold #2
  We have a total of 74111 backgrounds for training and 18528 for testing
  We're training on 74111 signals and testing on 18528 signals
  Fitting random forest classifier...


[Parallel(n_jobs=120)]: Done  63 out of 100 | elapsed:    1.1s remaining:    0.6s
[Parallel(n_jobs=120)]: Done 100 out of 100 | elapsed:    1.4s finished
[Parallel(n_jobs=100)]: Using backend ThreadingBackend with 100 concurrent workers.
[Parallel(n_jobs=100)]: Done   2 out of 100 | elapsed:    0.0s remaining:    1.0s
[Parallel(n_jobs=100)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=120)]: Using backend ThreadingBackend with 120 concurrent workers.


Average precision score: 0.9958098688897338
Fold #3
  We have a total of 74111 backgrounds for training and 18528 for testing
  We're training on 74111 signals and testing on 18528 signals
  Fitting random forest classifier...


...

[Parallel(n_jobs=120)]: Done  63 out of 100 | elapsed:    1.1s remaining:    0.7s
[Parallel(n_jobs=120)]: Done 100 out of 100 | elapsed:    1.3s finished
[Parallel(n_jobs=100)]: Using backend ThreadingBackend with 100 concurrent workers.
[Parallel(n_jobs=100)]: Done   2 out of 100 | elapsed:    0.0s remaining:    1.0s
[Parallel(n_jobs=100)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=120)]: Using backend ThreadingBackend with 120 concurrent workers.


Average precision score: 0.9952098594084503
Fold #5
  We have a total of 74112 backgrounds for training and 18527 for testing
  We're training on 74111 signals and testing on 18528 signals
  Fitting random forest classifier...


[Parallel(n_jobs=120)]: Done  63 out of 100 | elapsed:    1.1s remaining:    0.7s


Average precision score: 0.9944458799846088


[Parallel(n_jobs=120)]: Done 100 out of 100 | elapsed:    1.4s finished
[Parallel(n_jobs=100)]: Using backend ThreadingBackend with 100 concurrent workers.
[Parallel(n_jobs=100)]: Done   2 out of 100 | elapsed:    0.0s remaining:    1.0s
[Parallel(n_jobs=100)]: Done 100 out of 100 | elapsed:    0.0s finished


## Baseline performance

In [124]:
print(
    f"Average precision score (overall): mean={average_precision_scores.mean()} std={average_precision_scores.std()}"
)

Average precision score (overall): mean=0.9952313641292545 std=0.0004423752520697547


In [125]:
probs = clf.predict_proba(X)
assert isinstance(probs, np.ndarray)

[Parallel(n_jobs=100)]: Using backend ThreadingBackend with 100 concurrent workers.
[Parallel(n_jobs=100)]: Done   2 out of 100 | elapsed:    0.3s remaining:   13.6s
[Parallel(n_jobs=100)]: Done 100 out of 100 | elapsed:    0.8s finished


In [126]:
np.nonzero(probs[:, 1].round().astype(np.int32).astype(np.float64) != probs[:, 1])

(array([     32,      90,     167, ..., 2992631, 2992634, 2992635],
       shape=(534662,)),)

In [127]:
average_precision_score(y, probs[:, 1])

0.9140763043087048

In [128]:
from diquark.evaluation.metrics import calculate_counts_for_score_cuts

CROSS_SECTION_ATLAS_136_80 = {
    'BKG:gg_bbbar': 6.281E-04,
    'BKG:gg_ccbar': 6.362E-03,
    'BKG:gg_gg': 1.004E+07,
    'BKG:qqbar_gg': 3.639E-01,
    'BKG:gg_qqbar': 5.715E-02,
    'BKG:qg_qg': 3.058E+08,
    'BKG:qqbar_bbbar': 2.016E-05,
    'BKG:qqbar_ccbar': 2.013E-05,
    'BKG:qq_qq': 1.450E+09,
    'BKG:qqbar_qqbarNew': 6.053E-05,
    'BKG:qq_hbb_hadronic': 7.679E-11,
    'BKG:gg_hbb_hadronic': 6.318E-09,
    'BKG:gg_Hg': 4.014E-04,
    'BKG:qg_Hq_loop': 6.973E-03,
    'BKG:qqbar_Hg': 0,
    'BKG:ff_H': 0,
    'BKG:gg_H': 0,
    'BKG:ff_HZ': 1.447E-08,
    'BKG:ff_HW': 1.503E-08,
    'BKG:ff_Hff': 2.316E-02,
    'BKG:ff_Hff2': 3.986E-02,
    'BKG:gg_Httbar': 1.287E-07,
    'BKG:qq_Httbar': 1.448E-07,
    'BKG:qg_Hq': 1.411E-10,
    'BKG:ffbar_Wgm': 2.102E-05,
    'BKG:qg_Wj_hadronic': 9.486E-03,
    'BKG:qqbar_Wj_hadronic': 2.401E-03,
    'BKG:ff_ZW_bkg': 7.209E-06,
    'BKG:ff_WW_bkg': 2.738E-05,
    'BKG:ff_gmZgmZ_bkg': 2.298E-06,
    'BKG:gg_ttbar_bkg_hadronic': 8.018E-06,
    'BKG:qq_ttbar_bkg_hadronic': 9.390E-06,
    # Fixed cross section
    'SIG:Suu': 2.406E-03,
}

df_counts = calculate_counts_for_score_cuts(
    y,
    probs[:, 1],
    X["combined_invariant_mass"],
    df["label"],
    cross_sections=CROSS_SECTION_ATLAS_136_80,
    total_luminosity=3000,
    cuts=[0.0, 0.25, 0.5, 0.6, 0.75, 0.8, 0.9, 1.0],
    use_real_event_percentiles=False,
)

In [129]:
df_counts

,Process,0.0,0.25,0.5,0.6,0.75,0.8,0.9,1.0
0,BKG:ff_HW,1.228026e-05,1.228026e-05,1.228026e-05,1.228026e-05,1.228026e-05,1.228026e-05,3.288865e-06,0.0
1,BKG:ff_HZ,1.212919e-05,1.212919e-05,1.212919e-05,1.212919e-05,1.212919e-05,1.212919e-05,3.367748e-06,0.0
2,BKG:ff_Hff,2.552000e+00,2.552000e+00,2.552000e+00,2.552000e+00,2.552000e+00,2.552000e+00,1.278432e-01,0.0
3,BKG:ff_Hff2,3.657952e+00,3.657952e+00,3.657952e+00,3.657952e+00,3.657952e+00,3.657952e+00,1.602372e-01,0.0
4,BKG:ff_WW_bkg,2.736823e-02,2.736823e-02,2.736823e-02,2.736823e-02,2.736823e-02,2.736823e-02,1.841661e-02,0.0
5,BKG:ff_ZW_bkg,1.458741e-03,1.458741e-03,1.458741e-03,1.458741e-03,1.458741e-03,1.458741e-03,3.200796e-04,0.0
6,BKG:ff_gmZgmZ_bkg,9.583349e-04,9.583349e-04,9.583349e-04,9.583349e-04,9.583349e-04,9.583349e-04,4.216370e-04,0.0
7,BKG:ffbar_Wgm,4.470323e-03,4.470323e-03,4.470323e-03,4.470323e-03,4.470323e-03,4.470323e-03,1.906934e-03,0.0
8,BKG:gg_Hg,3.997944e-03,3.997944e-03,3.997944e-03,3.997944e-03,3.997944e-03,3.997944e-03,7.345620e-04,0.0
9,BKG:gg_Httbar,2.453859e-04,2.453859e-04,2.453859e-04,2.453859e-04,2.453859e-04,2.453859e-04,1.174593e-04,0.0


In [130]:
from sklearn.model_selection import GridSearchCV

In [131]:
param_grid = [
    {"criterion": ["gini"], "n_estimators": [10, 50, 100, 200, 300, 400]},
    {"criterion": ["entropy"], "n_estimators": [10, 50, 100, 200, 300, 400]},
]

clf = RandomForestClassifier(n_jobs=-1, verbose=0)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=i)

print("Doing nested cross-validation with hyperparameter optimization")

# Grid search CV with parameter optimization
gsc = GridSearchCV(estimator=clf, param_grid=param_grid, cv=skf, verbose=2)
gsc.fit(X_balanced, y_balanced)
cv_results = gsc.cv_results_

Doing nested cross-validation with hyperparameter optimization
Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END ....................criterion=gini, n_estimators=10; total time=   1.1s
[CV] END ....................criterion=gini, n_estimators=10; total time=   1.0s
[CV] END ....................criterion=gini, n_estimators=10; total time=   1.0s
[CV] END ....................criterion=gini, n_estimators=10; total time=   1.1s
[CV] END ....................criterion=gini, n_estimators=10; total time=   0.9s
[CV] END ....................criterion=gini, n_estimators=50; total time=   1.2s
[CV] END ....................criterion=gini, n_estimators=50; total time=   1.3s
[CV] END ....................criterion=gini, n_estimators=50; total time=   1.2s
[CV] END ....................criterion=gini, n_estimators=50; total time=   1.2s
[CV] END ....................criterion=gini, n_estimators=50; total time=   1.3s
[CV] END ...................criterion=gini, n_estimators=100; tota

In [132]:
cv_results

{'mean_fit_time': array([0.98844929, 1.19044838, 1.39229851, 2.30638714, 3.1594615 ,
        3.87391138, 0.78256779, 1.05905285, 1.27608218, 1.96724882,
        2.64489055, 3.30145097]),
 'std_fit_time': array([0.07150899, 0.03372254, 0.04949111, 0.03925637, 0.05457396,
        0.06101224, 0.041127  , 0.04638046, 0.07955688, 0.0249355 ,
        0.05230263, 0.07659923]),
 'mean_score_time': array([0.01912231, 0.03483205, 0.05799131, 0.07738929, 0.12727304,
        0.10425973, 0.01712503, 0.03238111, 0.05442309, 0.06994071,
        0.12041683, 0.10376787]),
 'std_score_time': array([0.00194135, 0.0018219 , 0.00192156, 0.00613439, 0.06712492,
        0.00656053, 0.00101854, 0.00104032, 0.00239102, 0.00220869,
        0.06051106, 0.00681593]),
 'param_criterion': masked_array(data=['gini', 'gini', 'gini', 'gini', 'gini', 'gini',
                    'entropy', 'entropy', 'entropy', 'entropy', 'entropy',
                    'entropy'],
              mask=[False, False, False, False, False, F